# [2] Naive Trial with Qwen-3 just using Prompt

## Imports

In [ ]:
import env

In [ ]:
from epidec.models.qwen3 import ChatHistory, Qwen3Model
from epidec.datasets import SWUnivDaconDataset

from torch.utils.data import DataLoader

import pandas as pd
from tqdm.auto import tqdm

import json
import sys

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = SWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = SWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Dataloaders

In [ ]:
BATCH_SIZE = 1, 1, 1

#train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE[0], shuffle=True)
#valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE[1], shuffle=True)
#test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE[2], shuffle=False)

## Define Model

In [ ]:
query = lambda p: f"""Analyze the following Korean text paragraph to determine if it was written by a human or generated by AI:

**Text:**: {p}"""

In [ ]:
system_prompt = """You are an expert in distinguishing between human-written and AI-generated text. You specialize in detecting AI-generated content by analyzing **multi-domain token mixing patterns** - a key characteristic where AI models inconsistently blend vocabulary and expressions from different domains without proper contextual awareness.

## Core Detection Principle
AI language models tend to mechanically combine **cross-domain linguistic patterns** in unnatural ways, creating subtle inconsistencies that differ from human writing. The key is distinguishing between **natural domain-appropriate writing** (like formal Wikipedia style) and **artificial pattern mixing** across incompatible domains.

## Analysis Framework

### 1. Cross-Domain Pattern Mixing (Primary Focus)
- **Inappropriate Domain Combinations**: Detect mixing of linguistic patterns from fundamentally incompatible contexts (academic research + casual blog + news reporting style within same paragraph)
- **Unnatural Register Shifts**: Identify sudden shifts between domain-specific linguistic registers without contextual justification
- **Template Collision**: Spot evidence of multiple domain templates being artificially merged

### 2. Domain-Appropriate vs Artificial Patterns
- **Natural Domain Writing**: Recognize that structured, formal writing (Wikipedia, news, academic) is naturally systematic and should NOT be flagged as AI
- **Artificial Mixing Signals**: Look for inappropriate combination of patterns from incompatible domains (technical manual style + personal diary tone + academic hedging)
- **Korean Language Considerations**: Understand that repetitive formal endings ("~이며", "~이다", "~있다") are natural in Korean encyclopedic writing

### 3. Genuine AI Indicators (Focus Here)
- **Micro-level Inconsistencies**: Subtle mixing of grammatical patterns from different domains within single sentences
- **Unnatural Precision Gradients**: Alternating between overly precise technical language and overly casual expressions inappropriately
- **Cross-domain Leakage**: Academic uncertainty markers in news writing, casual discourse markers in formal contexts, etc.

### 3. AI-Characteristic Patterns
- **Over-precision**: Excessively perfect grammar and structure beyond natural human variation
- **Generalization Markers**: Overuse of qualifying terms like "generally," "typically," "most"
- **Balanced Structure**: Unnaturally symmetric or overly balanced paragraph construction

## Analysis Process

### Step 1: Domain Context Assessment
First establish the expected domain and genre of the text. Do NOT flag domain-appropriate formal patterns (Wikipedia formality, news structure, academic style) as AI indicators.

### Step 2: Cross-Domain Leakage Detection
Look specifically for inappropriate mixing of linguistic patterns from incompatible domains - this is the primary AI signal.

### Step 3: Micro-Pattern Analysis
Examine sentence-level and clause-level mixing of grammatical patterns that wouldn't naturally occur together in human writing.

## Output Format
You must respond with a valid JSON object in the following format:

```json
{
  "domain_context": "Assessment of expected domain and appropriateness of formal patterns",
  "cross_domain_leakage": "Analysis of inappropriate mixing between incompatible domain patterns",
  "micro_inconsistencies": "Subtle sentence-level grammatical pattern mixing detection",
  "detection_rationale": "Key evidence for cross-domain pattern mixing vs natural domain-appropriate writing",
  "probability": 0.75
}
```

**Critical Requirements:**
- Always output valid JSON format
- Probability must be a number between 0.0 and 1.0
- All text fields should be concise but informative
- Do not include any text outside the JSON object
- Ensure proper JSON escaping for quotes and special characters

## Important Considerations
- **Korean Wikipedia Style**: Formal, structured writing with repetitive endings is NATURAL and should not be flagged as AI
- **Genre-Appropriate Formality**: News, academic, and encyclopedic texts are naturally systematic
- **Focus on Cross-Domain Mixing**: The key AI signal is inappropriate pattern mixing between incompatible domains, not domain-appropriate formal writing
- **Micro-Level Analysis**: Look for subtle grammatical inconsistencies rather than obvious structural patterns
- **Avoid Over-Detection**: Structured, formal, or repetitive writing is often human-authored and domain-appropriate"""

In [ ]:
class ChatHistory(ChatHistory):
    def create_prompt(self, system_prompt: str, user_prompt: str = ""):
        return [dict(role="system", content=system_prompt), *self, dict(role="user", content=query(user_prompt))]

In [ ]:
class Qwen3ModelForTextClassification(Qwen3Model):
    context_length = 4096

    def classify(
        self,
        user_prompt: str,
        grammar: str | None = None,
        temperature: float = 0.6,
        top_p: float = 0.95,
        top_k: int = 20,
        min_p: float = 0,
        typical_p: float = 1.0,
        repeat_penalty: float = 1.0
    ) -> str:
        return "".join(self.chat(
            chat_history=ChatHistory(),
            user_prompt=user_prompt,
            system_prompt="/nothink " + system_prompt,
            tools=[],
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            min_p=min_p,
            typical_p=typical_p,
            stream=True,
            max_new_tokens=0,
            repeat_penalty=repeat_penalty,
            print_output=True,
            grammar=grammar
        ))

    @staticmethod
    def extract_json(response_text: str) -> int:
        try:
            return json.loads(response_text.split("</think>")[-1].strip().replace("```json", "").replace("```", ""))
        except Exception:
            return {}

    def validate(self, dataset: list, retry_count: int = 1, shuffle: bool = False):
        corrects, errors, true_human, false_human, results = [], [], [], [], []
        progress = tqdm(DataLoader(dataset, batch_size=BATCH_SIZE[1], shuffle=shuffle), desc="Validating...")

        for idx, data in enumerate(progress):
            label = data[1]
            data = data[0]

            for trial in range(retry_count):
                assistant_reply = self.classify(data)
                predicted = self.extract_json(assistant_reply)
                if predicted: break
                print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

            result = dict(question=data, label=label, predicted=predicted)
            predicted_label = 1 if predicted['probability'] >= 0.5 else 0
            if predicted_label == label:
                corrects.append(result)
                print(f"INFO: Correct prediction for index {idx}\n\n")
                if label == 0:
                    true_human.append(result)
            else:
                result = dict(**result, traceback=assistant_reply)
                errors.append(result)
                print(f"ERROR: Incorrect prediction for index {idx}\n\n")
                if label == 0:
                    false_human.append(result)
            results.append(result)
            progress.set_description(f"Correct: {len(corrects)}/{len(results)} [H: {len(true_human)}, A: {len(corrects)-len(true_human)}], Errors: {len(errors)}/{len(results)} [H: {len(false_human)}, A: {len(errors)-len(false_human)}]")

        print(f"INFO: Correct: {len(corrects)}/{len(results)}, Errors: {len(errors)}/{len(results)}")
        return corrects, errors, results

    def test(self, dataset: list | str, retry_count: int = 100):
        results = []
        if isinstance(dataset, str):
            dataset = [dict(question=dataset)]  # Wrap single string input in dict format

        for idx, data in enumerate(tqdm(dataset, desc="Testing...")):
            data = data[0]

            for trial in range(retry_count):
                assistant_reply = self.classify(data)
                predicted = self.extract_json(assistant_reply)
                if predicted: break
                print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

            result = dict(question=data, label=predicted['probability'])
            results.append(result)
        return results

In [ ]:
model = Qwen3ModelForTextClassification()

## Evaluation

In [ ]:
# Validation
corrects, errors, results = model.validate(dataset=valid_dataset, shuffle=True, retry_count=1)
pd.DataFrame(results)

In [ ]:
# Test
results = model.test(dataset=test_dataset)
pd.DataFrame(results)